# 03 — Proof

The acceptance check. It resets the tables, runs each batch, and prints one row per measurement. It does not read `DATA_QUALITY_MONITORING_RESULTS` and it does not wait for a schedule.

`PASS` means the expectation held. `FAIL` means a bad batch was outside the threshold. The column is that comparison. It is not flipped to mean "the check worked."

A good batch is `PASS`. The replay, the empty load, the spike, and the stale timestamp are `FAIL`.

How each metric is measured:

- **Pair duplicates.** `SYSTEM$DATA_METRIC_SCAN` with `DUPLICATE_COUNT` and `ARGUMENT_NAME => 'order_id, line_no'` returns the failing rows, and the proof counts them. A direct call cannot take more than one column. The good batch expects 0. The replay expects a number other than 0.
- **`order_id` alone is a contrast, not the check.** Each order has five lines, so `DUPLICATE_COUNT(order_id)` on every line is already non-zero on a good batch. The function also rejects a `WHERE` inside its argument and rejects a subquery, so line 1 is copied into a temporary table first. Line 1 is 0 on the good batch and non-zero after the replay, which is why the contrast is worth printing. It is still not the pair.
- **Row count.** `ROW_COUNT` cannot be called directly, and `SYSTEM$DATA_METRIC_SCAN` does not accept it. The proof is `COUNT(*)` on `bt_orders_latest_batch`, the same row set the scheduled DMF reads. Those rows are labeled as a count of the view, not as a DMF call.
- **Freshness.** A direct call of `SNOWFLAKE.CORE.FRESHNESS`. The stale batch inserts `load_ts` 3600 seconds in the past and the function is called immediately. There is no sleep.

In [ ]:
USE ROLE BT_BUILDER_ROLE;
USE WAREHOUSE BT_WH;
USE DATABASE BT_DQ_REF;
USE SCHEMA BT_DQ_REF;

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE bt_proof (
  check_name   VARCHAR,
  failure_mode VARCHAR,
  script       VARCHAR,
  metric       VARCHAR,
  observed     NUMBER(38, 0),
  threshold    VARCHAR,
  result       VARCHAR
);

## Reset

The counter restarts at 0. Ids are monotonic within this run.

In [ ]:
BEGIN;
DELETE FROM bt_orders;
DELETE FROM bt_load_ledger;
UPDATE bt_batch_id SET id = 0;
COMMIT;

## Good batch

1000 rows, unique `(order_id, line_no)`. Every check should be `PASS`.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1,
  MOD(ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, 5) + 1,
  'SKU-' || LPAD(TO_VARCHAR(FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1), 6, '0'),
  1,
  CURRENT_TIMESTAMP(),
  (SELECT id FROM bt_batch_id)
FROM TABLE(GENERATOR(ROWCOUNT => 1000));
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'good', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 1000, 'synthetic load'
FROM bt_batch_id;
COMMIT;

`DUPLICATE_COUNT` accepts a query over a table, not a subquery and not a `WHERE` inside the argument. Copying line 1 into a temporary table gives the one-column contrast a single column to read.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE bt_orders_line1 AS
SELECT order_id FROM bt_orders WHERE line_no = 1;

In [ ]:
INSERT INTO bt_proof
SELECT
  'duplicate_keys_order_id',
  'none (good batch, line_no = 1 only, so order_id is unique)',
  'good batch in 03_prove',
  'DUPLICATE_COUNT(order_id) on line_no = 1. Not the pair check.',
  SNOWFLAKE.CORE.DUPLICATE_COUNT(SELECT order_id FROM bt_orders_line1),
  'VALUE = 0 on this one-line slice',
  IFF(SNOWFLAKE.CORE.DUPLICATE_COUNT(SELECT order_id FROM bt_orders_line1) = 0, 'PASS', 'FAIL');

In [ ]:
INSERT INTO bt_proof
SELECT
  'duplicate_keys_pair',
  'none (good batch, unique (order_id, line_no))',
  'good batch in 03_prove',
  'SYSTEM$DATA_METRIC_SCAN DUPLICATE_COUNT on (order_id, line_no), row count',
  (SELECT COUNT(*) FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
      REF_ENTITY_NAME => 'BT_DQ_REF.BT_DQ_REF.bt_orders',
      METRIC_NAME => 'SNOWFLAKE.CORE.DUPLICATE_COUNT',
      ARGUMENT_NAME => 'order_id, line_no'))),
  'VALUE = 0',
  IFF((SELECT COUNT(*) FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
        REF_ENTITY_NAME => 'BT_DQ_REF.BT_DQ_REF.bt_orders',
        METRIC_NAME => 'SNOWFLAKE.CORE.DUPLICATE_COUNT',
        ARGUMENT_NAME => 'order_id, line_no'))) = 0, 'PASS', 'FAIL');

In [ ]:
INSERT INTO bt_proof
SELECT
  'latest_batch_rowcount',
  'none (good batch, 1000 rows)',
  'good batch in 03_prove',
  'ROW_COUNT on bt_orders_latest_batch (COUNT of the view)',
  (SELECT COUNT(*) FROM bt_orders_latest_batch),
  '800 <= VALUE AND VALUE <= 1200',
  IFF((SELECT COUNT(*) FROM bt_orders_latest_batch) BETWEEN 800 AND 1200, 'PASS', 'FAIL');

In [ ]:
INSERT INTO bt_proof
SELECT
  'freshness',
  'none (load_ts is the insert timestamp)',
  'good batch in 03_prove',
  'FRESHNESS(load_ts) direct call',
  SNOWFLAKE.CORE.FRESHNESS(SELECT load_ts FROM bt_orders_latest_freshness),
  'VALUE < 900',
  IFF(SNOWFLAKE.CORE.FRESHNESS(SELECT load_ts FROM bt_orders_latest_freshness) < 900, 'PASS', 'FAIL');

## Duplicate replay

The same `(order_id, line_no)` values, inserted again with a new `load_ts`. The pair check and the one-column contrast both move off zero, so both rows are `FAIL`.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT order_id, line_no, sku, qty,
       DATEADD('minute', 5, CURRENT_TIMESTAMP()),
       (SELECT id FROM bt_batch_id)
FROM bt_orders
WHERE batch_id = (SELECT MIN(batch_id) FROM bt_orders);
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'duplicate_replay', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 1000, 'synthetic load'
FROM bt_batch_id;
COMMIT;

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE bt_orders_line1 AS
SELECT order_id FROM bt_orders WHERE line_no = 1;

In [ ]:
INSERT INTO bt_proof
SELECT
  'duplicate_keys_order_id',
  'replay of line_no = 1. Still not the pair.',
  'duplicate_replay batch in 03_prove',
  'DUPLICATE_COUNT(order_id) on line_no = 1. Not the pair check.',
  SNOWFLAKE.CORE.DUPLICATE_COUNT(SELECT order_id FROM bt_orders_line1),
  'VALUE = 0 on this one-line slice',
  IFF(SNOWFLAKE.CORE.DUPLICATE_COUNT(SELECT order_id FROM bt_orders_line1) = 0, 'PASS', 'FAIL');

In [ ]:
INSERT INTO bt_proof
SELECT
  'duplicate_keys_pair',
  'same (order_id, line_no) inserted again with a new load_ts',
  'duplicate_replay batch in 03_prove',
  'SYSTEM$DATA_METRIC_SCAN DUPLICATE_COUNT on (order_id, line_no), row count',
  (SELECT COUNT(*) FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
      REF_ENTITY_NAME => 'BT_DQ_REF.BT_DQ_REF.bt_orders',
      METRIC_NAME => 'SNOWFLAKE.CORE.DUPLICATE_COUNT',
      ARGUMENT_NAME => 'order_id, line_no'))),
  'VALUE = 0',
  IFF((SELECT COUNT(*) FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
        REF_ENTITY_NAME => 'BT_DQ_REF.BT_DQ_REF.bt_orders',
        METRIC_NAME => 'SNOWFLAKE.CORE.DUPLICATE_COUNT',
        ARGUMENT_NAME => 'order_id, line_no'))) = 0, 'PASS', 'FAIL');

## Empty attempt

Deletes the table and still writes a ledger row. The latest-batch view reads the ledger, so it returns 0 and the row-count check is `FAIL`. Reading `MAX(batch_id)` off `bt_orders` would have skipped this attempt.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'empty', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 0, 'synthetic load'
FROM bt_batch_id;
COMMIT;

In [ ]:
INSERT INTO bt_proof
SELECT
  'latest_batch_rowcount',
  'empty load deletes the table; ledger still records the attempt',
  'empty batch in 03_prove',
  'ROW_COUNT on bt_orders_latest_batch (COUNT of the view)',
  (SELECT COUNT(*) FROM bt_orders_latest_batch),
  '800 <= VALUE AND VALUE <= 1200',
  IFF((SELECT COUNT(*) FROM bt_orders_latest_batch) BETWEEN 800 AND 1200, 'PASS', 'FAIL');

## Spike

10000 rows, same 5-line shape. Outside 800 to 1200, so `FAIL`.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1,
  MOD(ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, 5) + 1,
  'SKU-' || LPAD(TO_VARCHAR(FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1), 6, '0'),
  1,
  CURRENT_TIMESTAMP(),
  (SELECT id FROM bt_batch_id)
FROM TABLE(GENERATOR(ROWCOUNT => 10000));
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'spike', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 10000, 'synthetic load'
FROM bt_batch_id;
COMMIT;

In [ ]:
INSERT INTO bt_proof
SELECT
  'latest_batch_rowcount',
  'bounded spike, 10000 rows',
  'spike batch in 03_prove',
  'ROW_COUNT on bt_orders_latest_batch (COUNT of the view)',
  (SELECT COUNT(*) FROM bt_orders_latest_batch),
  '800 <= VALUE AND VALUE <= 1200',
  IFF((SELECT COUNT(*) FROM bt_orders_latest_batch) BETWEEN 800 AND 1200, 'PASS', 'FAIL');

## Stale timestamp

A fresh 1000-row batch whose `load_ts` is written 3600 seconds in the past. The freshness call happens immediately. There is no sleep and no procedure.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1,
  MOD(ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, 5) + 1,
  'SKU-' || LPAD(TO_VARCHAR(FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1), 6, '0'),
  1,
  DATEADD('second', -3600, CURRENT_TIMESTAMP()),
  (SELECT id FROM bt_batch_id)
FROM TABLE(GENERATOR(ROWCOUNT => 1000));
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'stale_fixture', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 1000,
       'load_ts written 3600 seconds old; not a wait'
FROM bt_batch_id;
COMMIT;

In [ ]:
INSERT INTO bt_proof
SELECT
  'freshness',
  'load_ts inserted 3600 seconds in the past',
  'stale_fixture batch in 03_prove',
  'FRESHNESS(load_ts) direct call',
  SNOWFLAKE.CORE.FRESHNESS(SELECT load_ts FROM bt_orders_latest_freshness),
  'VALUE < 900',
  IFF(SNOWFLAKE.CORE.FRESHNESS(SELECT load_ts FROM bt_orders_latest_freshness) < 900, 'PASS', 'FAIL');

## Results

Eight rows. The four good-batch rows are `PASS`. The replay, the empty load, the spike, and the stale timestamp are `FAIL`.

In [ ]:
SELECT check_name, failure_mode, metric, observed, threshold, result
FROM bt_proof
ORDER BY check_name, failure_mode;

In [ ]:
SELECT batch_id, batch_name, row_count, note
FROM bt_load_ledger
ORDER BY batch_id;